# Form 144 API: see insider selling before the Form 4

A Form 144 is filed before an insider sells restricted or control stock. Query them by size, company and insider, with Python and real responses.

Read it with charts at https://www.quantisnow.com/sec-api/guides/form-144-insider-selling. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

An affiliate who wants to sell more than 5,000 shares or $50,000 of stock in three months files a Form 144 on the day they place the order. The Form 4 that reports the sale comes up to two business days after it. So a Form 144 is the earliest public sign that an executive, director or large holder is selling.

In the API, Form 144s are in the [insider trades dataset](https://www.quantisnow.com/sec-api/insider-trading) with `formType:144`. They carry `plannedSale.shares` and `plannedSale.valueUsd`, the size and market value stated on the form, and no `purchased` or `sold` amounts, because nothing has traded yet.

## Planned sales over $10 million this month

### Which insiders filed to sell more than $10 million of stock this month?

A Form 144 comes before the sale, so it shows selling a few days before the Form 4 does.

```shell
curl -X POST https://www.quantisnow.com/api/v1/insider-trades \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "formType:144 AND plannedSale.valueUsd:>10000000 AND filedAt:[2026-11-01 TO 2026-11-24]",
  "from": 0,
  "size": 5,
  "sort": [{"plannedSale.valueUsd": {"order": "desc"}}]
}'
```

Response (real, first 2 of 41):

```json
{
  "total": {
    "value": 41,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0001974078-26-000416:0001206587",
      "accessionNo": "0001974078-26-000416",
      "formType": "144",
      "amendment": false,
      "restatesEarlier": false,
      "filedAt": "2026-11-13T16:28:51-05:00",
      "transactionDate": "2026-11-13",
      "issuer": {
        "cik": "0001660280",
        "name": "Tenable Holdings, Inc.",
        "ticker": "TENB"
      },
      "reportingOwner": {
        "cik": "0001206587",
        "name": "COVIELLO ARTHUR W JR",
        "officerTitle": null,
        "isDirector": true,
        "isOfficer": false,
        "isTenPercentOwner": false
      },
      "transactionCodes": [],
      "purchased": {
        "shares": 0,
        "valueUsd": 0
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": true,
      "plannedSale": {
        "shares": 12000,
        "valueUsd": 4100000000
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1660280/000197407826000416/0001974078-26-000416-index.htm"
    },
    {
      "id": "0001969223-26-001210:0001622058",
      "accessionNo": "0001969223-26-001210",
      "formType": "144",
      "amendment": false,
      "restatesEarlier": false,
      "filedAt": "2026-11-23T16:53:25-05:00",
      "transactionDate": "2026-11-23",
      "issuer": {
        "cik": "0001668397",
        "name": "Medpace Holdings, Inc.",
        "ticker": "MEDP"
      },
      "reportingOwner": {
        "cik": "0001622058",
        "name": "Troendle August J.",
        "officerTitle": null,
        "isDirector": false,
        "isOfficer": false,
        "isTenPercentOwner": false
      },
      "transactionCodes": [],
      "purchased": {
        "shares": 0,
        "valueUsd": 0
      },
      "sold": {
        "shares": 0,
        "valueUsd": 0
      },
      "sharesOwnedAfter": null,
      "rule10b51": false,
      "plannedSale": {
        "shares": 338225,
        "valueUsd": 236757500
      },
      "linkToFilingIndex": "https://www.sec.gov/Archives/edgar/data/1668397/000196922326001210/0001969223-26-001210-index.htm"
    }
  ]
}
```

A Form 144 states an intent, not a sale. The seller can sell less, or nothing. Match it to the Form 4 that follows by issuer and `reportingOwner.cik` to see what actually sold.

## Match each Form 144 to the sale that followed

In [ ]:
import os, requests

BASE = "https://www.quantisnow.com/api/v1"
KEY = os.environ["QUANTISNOW_API_KEY"]

def search(endpoint, query, size=50, sort=None, start=0):
    body = {"query": query, "size": size, "from": start}
    if sort:
        body["sort"] = sort
    res = requests.post(f"{BASE}/{endpoint}", json=body,
                        headers={"Authorization": f"Bearer {KEY}"}, timeout=30)
    res.raise_for_status()
    return res.json()

planned = search("insider-trades",
                 "formType:144 AND plannedSale.valueUsd:>1000000 AND filedAt:[2026-11-01 TO 2026-11-23]",
                 sort="plannedSale.valueUsd:desc")["data"]

for p in planned:
    who, ticker = p["reportingOwner"]["cik"], p["issuer"]["ticker"]
    sales = search("insider-trades",
                   f'formType:4 AND reportingOwner.cik:{who} AND issuer.ticker:{ticker} '
                   f'AND sold.valueUsd:>0 AND filedAt:>={p["filedAt"][:10]}', size=10)["data"]
    sold = sum(s["sold"]["valueUsd"] for s in sales)
    print(ticker, p["reportingOwner"]["name"], round(p["plannedSale"]["valueUsd"]), "->", round(sold))

## In JavaScript

```javascript
const BASE = "https://www.quantisnow.com/api/v1";

async function search(endpoint, query, { size = 50, sort, from = 0 } = {}) {
  const res = await fetch(BASE + "/" + endpoint, {
    method: "POST",
    headers: {
      Authorization: "Bearer " + process.env.QUANTISNOW_API_KEY,
      "Content-Type": "application/json",
    },
    body: JSON.stringify({ query, size, from, sort }),
  });
  if (!res.ok) throw new Error((await res.json()).error.message);
  return res.json();
}

const page = await search(
  "insider-trades",
  "formType:144 AND issuer.ticker:NVDA AND filedAt:[2026-08-01 TO 2026-11-23]",
  { sort: "filedAt:desc" },
);
console.log(page.total.value, "planned sales");
```